In [1]:
# Ops Module
# to plot parquet files

## CORE varaibles
# var_0 = icesat; var_1 and var_2 = radar/flight lines

# lat, lon or polar stereo geom
# time_0 _1 _2 = time of acquisition
# angle = by default between 1,2 paths for radar only (heading values for future)

# elevation_surface_0 _1 _2 = meters (WGS84)
# elevation_bottom_0 _1 _2 = meters 
# power_surface_1 _2 = dB 
# power_bottom_1 _2 = dB 

In [ ]:
# imports and inits
import holoviews as hv
import hvplot.pandas
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs
import panel as pn
import numpy as np
import pandas as pd
import geopandas as gpd

hvplot.extension('bokeh')
pn.extension()  # 'bokeh' is not a valid pn extension name — drop it

# load data
df = gpd.read_parquet('../data/icesat-radar-merged_points_power_extracted.parquet')
icesat_df = gpd.read_parquet('../data/atl06_land_ice_heights.parquet')

# dataset overview: column names, types, shape, and first 4 values
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'shape': [(len(df),) for _ in df.columns],
    'first 4 values': [df[col].head(4).tolist() for col in df.columns]
})
summary.index.name = 'column'
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
summary

Shape: 106 rows x 24 columns



,dtype,shape,first 4 values
column,,,
geometry,geometry,"(106,)",[POINT Z (-1416146.3490113455 -466316.07374948...
index_right,int64,"(106,)","[3104, 2982, 405, 5959]"
id_1,str,"(106,)","[Data_20100103_02_016, Data_20100103_02_009, D..."
id_2,str,"(106,)","[Data_20100103_02_009, Data_20100103_02_016, D..."
collection_1,str,"(106,)","[2009_Antarctica_TO, 2009_Antarctica_TO, 2009_..."
collection_2,str,"(106,)","[2009_Antarctica_TO, 2009_Antarctica_TO, 2014_..."
geom_1,geometry,"(106,)",[LINESTRING (-1421480.5080355064 -467364.09767...
geom_2,geometry,"(106,)",[LINESTRING (-1420367.7360609265 -447002.17230...
opr:date_1,str,"(106,)","[20100103, 20100103, 20091102, 20141121]"


In [ ]:
# ── Project to EPSG:3031 (Antarctic polar stereographic) ──────────────────────
pts = df.to_crs('EPSG:3031').copy()
pts['x'] = pts.geometry.x
pts['y'] = pts.geometry.y
pts_df = pd.DataFrame(pts.drop(columns=['geometry', 'geom_1', 'geom_2']))

# ── Deduplicated flight lines ─────────────────────────────────────────────────
lines_a = (df.set_geometry('geom_1').to_crs('EPSG:3031')
             [['id_1', 'collection_1', 'geom_1']].drop_duplicates('id_1')
             .rename(columns={'id_1': 'id', 'collection_1': 'collection', 'geom_1': 'geometry'})
             .set_geometry('geometry'))
lines_b = (df.set_geometry('geom_2').to_crs('EPSG:3031')
             [['id_2', 'collection_2', 'geom_2']].drop_duplicates('id_2')
             .rename(columns={'id_2': 'id', 'collection_2': 'collection', 'geom_2': 'geometry'})
             .set_geometry('geometry'))
all_lines = gpd.GeoDataFrame(pd.concat([lines_a, lines_b], ignore_index=True)).drop_duplicates('id')

# ── Natural Earth Antarctic ice shelves (downloaded once by cartopy) ──────────
import cartopy.io.shapereader as shpreader
_ice_path = shpreader.natural_earth(
    resolution='50m', category='physical', name='antarctic_ice_shelves_polys')
_ice_gdf = gpd.read_file(_ice_path).to_crs('EPSG:3031')
_ice_layer = _ice_gdf.hvplot(
    color='#dce9f5', line_color='steelblue', line_width=0.5,
    label='Ice shelves', geo=False)

# ── Background styles — Natural Earth via geoviews.feature, pre-rendered once ─
_proj = dict(projection=ccrs.SouthPolarStereo(), scale='50m')
_BACKGROUNDS = {
    'Ocean + Coast': (
        gf.ocean.opts(**_proj) *
        gf.coastline.opts(**_proj)
    ),
    'Land + Coast': (
        gf.land.opts(**_proj) *
        gf.coastline.opts(**_proj)
    ),
    'Ice shelves + Coast': (
        gf.land.opts(**_proj) *
        _ice_layer *
        gf.coastline.opts(**_proj)
    ),
    'Coast only': (
        gf.coastline.opts(**_proj)
    ),
}

# ── Flight lines layer ────────────────────────────────────────────────────────
_flight_lines = all_lines.hvplot(color='gray', line_width=0.6, label='Flight lines')

# ── Auto-detect variable columns ──────────────────────────────────────────────
_skip = {'x', 'y', 'index_right',
         'opr:segment_1', 'opr:segment_2', 'opr:frame_1', 'opr:frame_2'}
numeric_cols  = [c for c in pts_df.select_dtypes(include=[np.number]).columns if c not in _skip]
category_cols = [c for c in ['collection_1', 'collection_2', 'opr:date_1', 'opr:date_2']
                 if c in pts_df.columns]
all_options   = ['(none — location only)'] + numeric_cols + category_cols

_CMAPS = {
    'crossing_angle':      'viridis',  'dist_m':              'plasma',
    'elev_diff':           'RdYlBu_r',
    'wgs84_1':             'plasma',   'wgs84_2':             'plasma',
    'power_surface_1':     'magma',    'power_surface_2':     'magma',
    'power_bottom_1':      'magma',    'power_bottom_2':      'magma',
    'elevation_surface_1': 'viridis',  'elevation_surface_2': 'viridis',
    'elevation_bottom_1':  'cividis',  'elevation_bottom_2':  'cividis',
}

# ── Widgets ───────────────────────────────────────────────────────────────────
default_var = 'crossing_angle' if 'crossing_angle' in all_options else all_options[0]

w_color = pn.widgets.Select(
    name='Variable', options=all_options, value=default_var, width=240)
w_cmap  = pn.widgets.Select(
    name='Colormap',
    options=['viridis', 'plasma', 'magma', 'cividis', 'RdYlBu_r', 'coolwarm', 'turbo', 'inferno'],
    value='viridis', width=200)
w_size  = pn.widgets.IntSlider(name='Point size', start=3, end=20, value=8, width=220)
w_bg    = pn.widgets.Select(
    name='Background', options=list(_BACKGROUNDS.keys()), value='Ocean + Coast', width=240)
w_lines = pn.widgets.Checkbox(name='Show flight lines', value=True)
w_alpha = pn.widgets.FloatSlider(
    name='Flight line opacity', start=0.05, end=0.6, step=0.05, value=0.2, width=220)

# ── Plot function ─────────────────────────────────────────────────────────────
def crossover_map(color_by, cmap, pt_size, bg_style, show_lines, line_alpha):
    hover = [c for c in pts_df.columns if c not in {'x', 'y', 'index_right'}]
    kw = dict(x='x', y='y', hover_cols=hover, label='Crossover points')

    if color_by == '(none — location only)':
        pts_plot = pts_df.hvplot.points(**kw, color='#e63946').opts(size=pt_size)
    elif color_by in category_cols:
        pts_plot = pts_df.hvplot.points(**kw, c=color_by).opts(size=pt_size)
    else:
        pts_plot = pts_df.hvplot.points(
            **kw, c=color_by, cmap=_CMAPS.get(color_by, cmap), colorbar=True).opts(size=pt_size)

    base = _BACKGROUNDS[bg_style]
    if show_lines:
        base = base * _flight_lines.opts(alpha=line_alpha)

    plot = (base * pts_plot).opts(
        frame_width=680, frame_height=680,
        active_tools=['pan', 'wheel_zoom'],
        title='Crossover Points')

    return pn.pane.HoloViews(plot, sizing_mode='scale_both', aspect_ratio=1)

# ── Layout ────────────────────────────────────────────────────────────────────
controls = pn.Column(
    pn.pane.Markdown("### Display options"),
    pn.pane.Markdown("**Color crossovers by variable:**"),
    w_color, w_cmap, w_size,
    pn.layout.Divider(),
    pn.pane.Markdown("**Background:**"),
    w_bg,
    pn.layout.Divider(),
    w_lines, w_alpha,
    width=270, margin=(10, 20, 10, 10))

pn.Row(
    controls,
    pn.bind(crossover_map, w_color, w_cmap, w_size, w_bg, w_lines, w_alpha),
    sizing_mode='stretch_width'
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'d5bec87c-3c64-45b5-8890-f74a2ac25a0e': {'version…

UnknownReferenceError: can't resolve reference '39520f64-e271-4990-a217-b05fb328ac79'

UnknownReferenceError: can't resolve reference '39520f64-e271-4990-a217-b05fb328ac79'

UnknownReferenceError: can't resolve reference '39520f64-e271-4990-a217-b05fb328ac79'

UnknownReferenceError: can't resolve reference '39520f64-e271-4990-a217-b05fb328ac79'

...

UnknownReferenceError: can't resolve reference '76773258-b3f3-463b-861a-ed690626afa5'

UnknownReferenceError: can't resolve reference 'a6488e4c-8a1b-4a44-b185-73153dc75d34'

UnknownReferenceError: can't resolve reference 'a6488e4c-8a1b-4a44-b185-73153dc75d34'

UnknownReferenceError: can't resolve reference 'a6488e4c-8a1b-4a44-b185-73153dc75d34'

UnknownReferenceError: can't resolve reference 'a6488e4c-8a1b-4a44-b185-73153dc75d34'